# Phase 5: Model Selection & Training

Train baseline and advanced models, evaluate on untouched test data, and save the best model by minority-class F1-score.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

BASE = Path.cwd()
if (BASE / 'notebooks').exists():
    BASE = BASE.parent
DATA = BASE / 'data'
FIG_DIR = BASE / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(DATA / 'X_train_scaled.csv')
X_test = pd.read_csv(DATA / 'X_test_scaled.csv')
y_train = pd.read_csv(DATA / 'y_train_resampled.csv').iloc[:, 0]
y_test = pd.read_csv(DATA / 'y_test.csv').iloc[:, 0]

print('Shapes:', X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print('Test class distribution:')
print(y_test.value_counts(normalize=True).rename('pct').mul(100).round(2))


In [ ]:
# Baseline: Logistic Regression
log_reg = LogisticRegression(max_iter=1200, solver='lbfgs', n_jobs=None, random_state=42)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

print('=== Logistic Regression: classification_report ===')
print(classification_report(y_test, y_pred_lr, digits=4))

cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr)
disp_lr.plot(values_format='d', cmap='Blues')
plt.title('Logistic Regression - Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Advanced model: Random Forest (CPU-efficient tuned settings)
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    min_samples_split=8,
    min_samples_leaf=4,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest: classification_report ===')
print(classification_report(y_test, y_pred_rf, digits=4))

cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
disp_rf.plot(values_format='d', cmap='Greens')
plt.title('Random Forest - Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Compare minority-class (IsReturned=1) F1 score and save best model
report_lr = classification_report(y_test, y_pred_lr, output_dict=True, zero_division=0)
report_rf = classification_report(y_test, y_pred_rf, output_dict=True, zero_division=0)

f1_lr = report_lr.get('1', {}).get('f1-score', 0.0)
f1_rf = report_rf.get('1', {}).get('f1-score', 0.0)

print(f'Logistic Regression minority F1: {f1_lr:.4f}')
print(f'Random Forest minority F1: {f1_rf:.4f}')

best_name = 'RandomForestClassifier' if f1_rf >= f1_lr else 'LogisticRegression'
best_model = rf if f1_rf >= f1_lr else log_reg

joblib.dump(best_model, DATA / 'best_model.joblib')
print('Best model:', best_name)
print('Saved:', DATA / 'best_model.joblib')
